In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Load the main dataset
df = pd.read_csv('Bangalore_Monthly_Final_Corrected.csv')

# Basic info
print("="*80)
print("DATASET OVERVIEW")
print("="*80)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Missing values analysis
print("\n" + "="*80)
print("MISSING VALUES ANALYSIS")
print("="*80)
missing = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2),
    'Data_Type': df.dtypes
})
missing = missing[missing['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False)
print(missing)

if len(missing) == 0:
    print("No missing values found!")

# Check for duplicate rows
print("\n" + "="*80)
print("DUPLICATE ROWS")
print("="*80)
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")
if duplicates > 0:
    print(f"Percentage: {duplicates/len(df)*100:.2f}%")

DATASET OVERVIEW
Shape: (23042, 26)
Columns: ['system:index', 'AOD', 'AtmosphericMoisture', 'BulkDensity', 'Clay', 'Copper', 'Elevation', 'Iron', 'LST', 'NDVI', 'NDWI', 'NO2', 'Nitrogen', 'Rain', 'SAVI', 'SO2', 'Sand', 'Slope', 'SoilMoisture', 'Temp', 'date', 'green_fraction', 'month', 'pH', 'year', '.geo']

Data types:
system:index               str
AOD                    float64
AtmosphericMoisture    float64
BulkDensity            float64
Clay                   float64
Copper                 float64
Elevation                int64
Iron                   float64
LST                    float64
NDVI                   float64
NDWI                   float64
NO2                    float64
Nitrogen               float64
Rain                   float64
SAVI                   float64
SO2                    float64
Sand                   float64
Slope                  float64
SoilMoisture           float64
Temp                   float64
date                       str
green_fraction         floa

In [3]:
# Statistical summary for numerical columns
print("\n" + "="*80)
print("STATISTICAL SUMMARY")
print("="*80)

# Identify numerical columns (excluding system:index, date, year, month, .geo)
exclude_cols = ['system:index', 'date', '.geo']
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Remove any excluded columns
numerical_cols = [col for col in numerical_cols if col not in exclude_cols]

print(df[numerical_cols].describe().T)

# Outlier detection using IQR method
print("\n" + "="*80)
print("OUTLIER DETECTION (IQR Method)")
print("="*80)

outlier_summary = []
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_count = len(outliers)
    outlier_pct = (outlier_count / len(df)) * 100
    
    outlier_summary.append({
        'Column': col,
        'Outlier_Count': outlier_count,
        'Outlier_Percentage': round(outlier_pct, 2),
        'Lower_Bound': round(lower_bound, 4),
        'Upper_Bound': round(upper_bound, 4),
        'Min_Value': round(df[col].min(), 4),
        'Max_Value': round(df[col].max(), 4)
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df = outlier_df[outlier_df['Outlier_Count'] > 0].sort_values('Outlier_Percentage', ascending=False)
print(outlier_df)

# Z-score based outlier detection (alternative method)
print("\n" + "="*80)
print("OUTLIER DETECTION (Z-Score Method, |z| > 3)")
print("="*80)

zscore_outliers = []
for col in numerical_cols:
    z_scores = np.abs(stats.zscore(df[col], nan_policy='omit'))
    outlier_count = (z_scores > 3).sum()
    outlier_pct = (outlier_count / len(df)) * 100
    
    zscore_outliers.append({
        'Column': col,
        'Outlier_Count': outlier_count,
        'Outlier_Percentage': round(outlier_pct, 2)
    })

zscore_df = pd.DataFrame(zscore_outliers)
zscore_df = zscore_df[zscore_df['Outlier_Count'] > 0].sort_values('Outlier_Percentage', ascending=False)
print(zscore_df)


STATISTICAL SUMMARY
                       count         mean        std          min  \
AOD                  23042.0     0.611650   0.210881     0.000000   
AtmosphericMoisture  23042.0    15.809468   3.286048     7.879718   
BulkDensity          23042.0     1.355212   0.018785     1.260000   
Clay                 23042.0    23.493165   3.026940    17.500000   
Copper               23042.0     0.568458   0.047124     0.449000   
Elevation            23042.0   870.131369  51.645446   699.000000   
Iron                 23042.0     1.174658   0.151347     0.875000   
LST                  23042.0    30.783232   3.418285     2.790000   
NDVI                 23042.0     0.191136   0.072527    -0.019721   
NDWI                 23042.0    -0.199243   0.079935    -0.557356   
NO2                  23042.0     0.000040   0.000016    -0.000002   
Nitrogen             23042.0    20.244441   3.573394    15.800000   
Rain                 23042.0    74.372139  78.131598     0.000000   
SAVI         

In [4]:
print("\n" + "="*80)
print("DATA QUALITY CHECKS")
print("="*80)

# Check for constant/near-constant columns
print("\n1. LOW VARIANCE FEATURES:")
variance_check = []
for col in numerical_cols:
    unique_count = df[col].nunique()
    unique_ratio = unique_count / len(df)
    std_dev = df[col].std()
    
    if unique_ratio < 0.01:  # Less than 1% unique values
        variance_check.append({
            'Column': col,
            'Unique_Count': unique_count,
            'Unique_Ratio': round(unique_ratio, 4),
            'Std_Dev': round(std_dev, 4)
        })

if variance_check:
    print(pd.DataFrame(variance_check))
else:
    print("No low variance features found.")

# Check for negative values where they shouldn't exist
print("\n2. NEGATIVE VALUES CHECK (for features that should be positive):")
positive_features = ['AOD', 'Rain', 'Temp', 'LST', 'green_fraction', 'BulkDensity', 
                     'Clay', 'Sand', 'Copper', 'Iron', 'Nitrogen', 'Elevation']
negative_values = []

for col in positive_features:
    if col in df.columns:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            negative_values.append({
                'Column': col,
                'Negative_Count': neg_count,
                'Negative_Percentage': round(neg_count/len(df)*100, 2),
                'Min_Value': round(df[col].min(), 4)
            })

if negative_values:
    print(pd.DataFrame(negative_values))
else:
    print("No unexpected negative values found.")

# Check value ranges for specific features
print("\n3. VALUE RANGE VALIDATION:")
range_checks = {
    'NDVI': (-1, 1),
    'NDWI': (-1, 1),
    'SAVI': (-1, 1),
    'pH': (0, 14),
    'green_fraction': (0, 1),
    'month': (0, 12)
}

range_issues = []
for col, (min_val, max_val) in range_checks.items():
    if col in df.columns:
        out_of_range = df[(df[col] < min_val) | (df[col] > max_val)]
        if len(out_of_range) > 0:
            range_issues.append({
                'Column': col,
                'Expected_Range': f"[{min_val}, {max_val}]",
                'Out_of_Range_Count': len(out_of_range),
                'Actual_Min': round(df[col].min(), 4),
                'Actual_Max': round(df[col].max(), 4)
            })

if range_issues:
    print(pd.DataFrame(range_issues))
else:
    print("All features are within expected ranges.")



DATA QUALITY CHECKS

1. LOW VARIANCE FEATURES:
        Column  Unique_Count  Unique_Ratio  Std_Dev
0  BulkDensity            14        0.0006   0.0188
1         Clay           129        0.0056   3.0269
2       Copper           187        0.0081   0.0471
3    Elevation           199        0.0086  51.6454
4         Iron           129        0.0056   0.1513
5     Nitrogen           128        0.0056   3.5734
6         Sand           187        0.0081   4.7124
7        month            35        0.0015  10.5563
8           pH             7        0.0003   0.0946
9         year             3        0.0001   0.7985

2. NEGATIVE VALUES CHECK (for features that should be positive):
No unexpected negative values found.

3. VALUE RANGE VALIDATION:
  Column Expected_Range  Out_of_Range_Count  Actual_Min  Actual_Max
0  month        [0, 12]               15670         0.0        35.0


In [5]:
print("\n" + "="*80)
print("CORRELATION ANALYSIS")
print("="*80)

# Calculate correlation matrix
corr_matrix = df[numerical_cols].corr()

# Find highly correlated features (|correlation| > 0.9)
print("\n1. HIGHLY CORRELATED FEATURES (|r| > 0.9):")
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            high_corr.append({
                'Feature_1': corr_matrix.columns[i],
                'Feature_2': corr_matrix.columns[j],
                'Correlation': round(corr_matrix.iloc[i, j], 4)
            })

if high_corr:
    print(pd.DataFrame(high_corr).sort_values('Correlation', ascending=False))
else:
    print("No highly correlated features found (|r| > 0.9).")

# Multicollinearity check (VIF is more comprehensive but this is quick)
# print("\n2. CORRELATION WITH TARGET (if applicable):")
# If you have a target variable, uncomment and replace 'target_col' with your target
# target_col = 'your_target_variable'
# if target_col in df.columns:
#     target_corr = df[numerical_cols].corrwith(df[target_col]).sort_values(ascending=False)
#     print(target_corr)



CORRELATION ANALYSIS

1. HIGHLY CORRELATED FEATURES (|r| > 0.9):
  Feature_1 Feature_2  Correlation
0      Clay      Iron       1.0000
1    Copper      Sand       1.0000
3      NDVI      SAVI       1.0000
5     month      year       0.9374
2      NDVI      NDWI      -0.9154
4      NDWI      SAVI      -0.9154


In [6]:
print("\n" + "="*80)
print("SPATIAL ANALYSIS")
print("="*80)

# Parse geo column to extract coordinates
import json

def extract_coordinates(geo_str):
    try:
        geo_dict = json.loads(geo_str)
        coords = geo_dict['coordinates']
        return coords[0], coords[1]  # longitude, latitude
    except:
        return None, None

df['longitude'], df['latitude'] = zip(*df['.geo'].apply(extract_coordinates))

print(f"Longitude range: [{df['longitude'].min():.6f}, {df['longitude'].max():.6f}]")
print(f"Latitude range: [{df['latitude'].min():.6f}, {df['latitude'].max():.6f}]")
print(f"Unique locations: {df[['longitude', 'latitude']].drop_duplicates().shape[0]}")

# Check for spatial outliers
print("\n1. POTENTIAL SPATIAL OUTLIERS:")
lon_z = np.abs(stats.zscore(df['longitude'].dropna()))
lat_z = np.abs(stats.zscore(df['latitude'].dropna()))
spatial_outliers = df[(lon_z > 3) | (lat_z > 3)]
print(f"Locations with unusual coordinates: {len(spatial_outliers)}")



SPATIAL ANALYSIS
Longitude range: [77.329012, 77.809610]
Latitude range: [12.656678, 13.218125]
Unique locations: 805

1. POTENTIAL SPATIAL OUTLIERS:
Locations with unusual coordinates: 0


In [7]:
print("\n" + "="*80)
print("FEATURE-SPECIFIC INSIGHTS")
print("="*80)

# Soil features
soil_features = ['Clay', 'Sand', 'pH', 'BulkDensity', 'Nitrogen', 'Copper', 'Iron']
available_soil = [f for f in soil_features if f in df.columns]
if available_soil:
    print("\n1. SOIL FEATURES:")
    print(df[available_soil].describe().T[['mean', 'std', 'min', 'max']])

# Climate features
climate_features = ['Rain', 'Temp', 'LST', 'AtmosphericMoisture', 'SoilMoisture']
available_climate = [f for f in climate_features if f in df.columns]
if available_climate:
    print("\n2. CLIMATE FEATURES:")
    print(df[available_climate].describe().T[['mean', 'std', 'min', 'max']])

# Vegetation indices
veg_indices = ['NDVI', 'NDWI', 'SAVI', 'green_fraction']
available_veg = [f for f in veg_indices if f in df.columns]
if available_veg:
    print("\n3. VEGETATION INDICES:")
    print(df[available_veg].describe().T[['mean', 'std', 'min', 'max']])

# Pollution indicators
pollution_features = ['AOD', 'NO2', 'SO2']
available_pollution = [f for f in pollution_features if f in df.columns]
if available_pollution:
    print("\n4. POLLUTION INDICATORS:")
    print(df[available_pollution].describe().T[['mean', 'std', 'min', 'max']])



FEATURE-SPECIFIC INSIGHTS

1. SOIL FEATURES:
                  mean       std     min     max
Clay         23.493165  3.026940  17.500  32.600
Sand         56.845834  4.712393  44.900  67.200
pH            6.482219  0.094606   6.200   6.800
BulkDensity   1.355212  0.018785   1.260   1.400
Nitrogen     20.244441  3.573394  15.800  50.100
Copper        0.568458  0.047124   0.449   0.672
Iron          1.174658  0.151347   0.875   1.630

2. CLIMATE FEATURES:
                          mean        std        min         max
Rain                 74.372139  78.131598   0.000000  403.165275
Temp                 23.183901   2.014681  19.461688   27.488935
LST                  30.783232   3.418285   2.790000   42.205000
AtmosphericMoisture  15.809468   3.286048   7.879718   20.641866
SoilMoisture          0.210931   0.072279   0.080776    0.390877

3. VEGETATION INDICES:
                    mean       std       min       max
NDVI            0.191136  0.072527 -0.019721  0.591292
NDWI           -

In [8]:
df['year']

0        2021
1        2021
2        2021
3        2021
4        2021
         ... 
23037    2023
23038    2023
23039    2023
23040    2023
23041    2023
Name: year, Length: 23042, dtype: int64

In [9]:
import pandas as pd
import numpy as np
from scipy import stats

# Load data
df = pd.read_csv('Bangalore_Monthly_Final_Corrected.csv')

print("Original shape:", df.shape)
print("Original columns:", df.columns.tolist())

# Drop perfectly correlated features
features_to_drop = ['Iron', 'Copper', 'SAVI']
df_cleaned = df.drop(columns=features_to_drop)

print("\n" + "="*80)
print("AFTER DROPPING REDUNDANT FEATURES")
print("="*80)
print(f"New shape: {df_cleaned.shape}")
print(f"Dropped: {features_to_drop}")
print(f"Remaining features: {df_cleaned.columns.tolist()}")

Original shape: (23042, 26)
Original columns: ['system:index', 'AOD', 'AtmosphericMoisture', 'BulkDensity', 'Clay', 'Copper', 'Elevation', 'Iron', 'LST', 'NDVI', 'NDWI', 'NO2', 'Nitrogen', 'Rain', 'SAVI', 'SO2', 'Sand', 'Slope', 'SoilMoisture', 'Temp', 'date', 'green_fraction', 'month', 'pH', 'year', '.geo']

AFTER DROPPING REDUNDANT FEATURES
New shape: (23042, 23)
Dropped: ['Iron', 'Copper', 'SAVI']
Remaining features: ['system:index', 'AOD', 'AtmosphericMoisture', 'BulkDensity', 'Clay', 'Elevation', 'LST', 'NDVI', 'NDWI', 'NO2', 'Nitrogen', 'Rain', 'SO2', 'Sand', 'Slope', 'SoilMoisture', 'Temp', 'date', 'green_fraction', 'month', 'pH', 'year', '.geo']


In [10]:
print("\n" + "="*80)
print("APPLYING TRANSFORMATIONS")
print("="*80)

# Create a copy for transformations
df_transformed = df_cleaned.copy()

# Parse date column first
df_transformed['date'] = pd.to_datetime(df_transformed['date'])

# Extract coordinates from .geo
import json

def extract_coordinates(geo_str):
    try:
        geo_dict = json.loads(geo_str)
        coords = geo_dict['coordinates']
        return coords[0], coords[1]  # longitude, latitude
    except:
        return None, None

df_transformed['longitude'], df_transformed['latitude'] = zip(
    *df_transformed['.geo'].apply(extract_coordinates)
)

print("\n1. SKEWNESS ANALYSIS (Before Transformation):")
print("-" * 60)

# Check skewness of numerical features
numerical_cols = ['AOD', 'AtmosphericMoisture', 'BulkDensity', 'Clay', 
                  'Elevation', 'LST', 'NDVI', 'NDWI', 'NO2', 'Nitrogen', 
                  'Rain', 'SO2', 'Sand', 'Slope', 'SoilMoisture', 'Temp', 
                  'green_fraction', 'pH']

skewness_before = {}
for col in numerical_cols:
    skew = df_transformed[col].skew()
    skewness_before[col] = skew
    
skew_df_before = pd.DataFrame(list(skewness_before.items()), 
                              columns=['Feature', 'Skewness_Before'])
skew_df_before = skew_df_before.sort_values('Skewness_Before', 
                                            key=abs, ascending=False)
print(skew_df_before)

print("\nFeatures with high skewness (|skew| > 1):")
high_skew = skew_df_before[abs(skew_df_before['Skewness_Before']) > 1]
print(high_skew)

# Apply transformations
print("\n2. APPLYING LOG TRANSFORMATIONS:")
print("-" * 60)

# Log transform for highly skewed features
# Rain - has zeros, use log1p
df_transformed['Rain_original'] = df_transformed['Rain']
df_transformed['Rain'] = np.log1p(df_transformed['Rain'])
print(f"✓ Rain: log1p transform applied (handles zeros)")

# NO2 - very small values, scale up then log
df_transformed['NO2_original'] = df_transformed['NO2']
df_transformed['NO2'] = np.log1p(df_transformed['NO2'] * 1e6)
print(f"✓ NO2: scaled (*1e6) then log1p transform")

# SO2 - very small values, scale up then log, handle negatives
df_transformed['SO2_original'] = df_transformed['SO2']
# Shift to make all positive first
so2_min = df_transformed['SO2'].min()
if so2_min < 0:
    df_transformed['SO2'] = np.log1p((df_transformed['SO2'] - so2_min) * 1e6)
else:
    df_transformed['SO2'] = np.log1p(df_transformed['SO2'] * 1e6)
print(f"✓ SO2: shifted & scaled, then log1p transform")

# Slope - right skewed
df_transformed['Slope_original'] = df_transformed['Slope']
df_transformed['Slope'] = np.log1p(df_transformed['Slope'])
print(f"✓ Slope: log1p transform applied")

# Check skewness after transformation
print("\n3. SKEWNESS ANALYSIS (After Transformation):")
print("-" * 60)

skewness_after = {}
for col in ['Rain', 'NO2', 'SO2', 'Slope']:
    skew = df_transformed[col].skew()
    skewness_after[col] = skew

comparison = pd.DataFrame({
    'Feature': ['Rain', 'NO2', 'SO2', 'Slope'],
    'Skewness_Before': [skewness_before[col] for col in ['Rain', 'NO2', 'SO2', 'Slope']],
    'Skewness_After': [skewness_after[col] for col in ['Rain', 'NO2', 'SO2', 'Slope']]
})
comparison['Improvement'] = abs(comparison['Skewness_Before']) - abs(comparison['Skewness_After'])
print(comparison)

print("\n4. TRANSFORMATION SUMMARY:")
print("-" * 60)
print(f"Original columns backed up with '_original' suffix")
print(f"Transformed features: Rain, NO2, SO2, Slope")
print(f"Coordinates extracted: longitude, latitude")
print(f"\nNew shape: {df_transformed.shape}")


APPLYING TRANSFORMATIONS

1. SKEWNESS ANALYSIS (Before Transformation):
------------------------------------------------------------
                Feature  Skewness_Before
13                Slope         3.921398
9              Nitrogen         3.237377
16       green_fraction         1.788646
2           BulkDensity        -1.351762
10                 Rain         1.125803
4             Elevation        -1.114143
8                   NO2         0.769811
0                   AOD         0.643766
3                  Clay         0.628245
1   AtmosphericMoisture        -0.613612
6                  NDVI         0.577661
5                   LST         0.493373
14         SoilMoisture         0.446126
7                  NDWI        -0.263111
11                  SO2         0.220652
12                 Sand        -0.104210
17                   pH        -0.077149
15                 Temp         0.056981

Features with high skewness (|skew| > 1):
           Feature  Skewness_Before
13      

/Users/absarkar/Developer/Capstone_Code/capstone_venv/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [11]:
print("\n" + "="*80)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*80)

# Prepare feature matrix (exclude non-predictive columns)
exclude_cols = ['system:index', 'date', '.geo', 'year', 'month',
                'Rain_original', 'NO2_original', 'SO2_original', 'Slope_original']

feature_cols = [col for col in df_transformed.columns if col not in exclude_cols]

print(f"\nAvailable features for modeling: {len(feature_cols)}")
print(feature_cols)

# Check for any remaining NaN after transformations
print("\n1. DATA QUALITY CHECK:")
print("-" * 60)
nan_check = df_transformed[feature_cols].isnull().sum()
if nan_check.sum() > 0:
    print("⚠ Features with NaN values:")
    print(nan_check[nan_check > 0])
else:
    print("✓ No NaN values in feature set")

# Correlation with potential targets
print("\n2. INTER-FEATURE CORRELATION MATRIX:")
print("-" * 60)
print("Top 10 strongest correlations (excluding self-correlation):\n")

# Calculate correlation matrix
corr_matrix = df_transformed[feature_cols].corr()

# Get upper triangle correlations
correlations = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        correlations.append({
            'Feature_1': corr_matrix.columns[i],
            'Feature_2': corr_matrix.columns[j],
            'Correlation': corr_matrix.iloc[i, j]
        })

corr_df = pd.DataFrame(correlations)
corr_df['Abs_Correlation'] = abs(corr_df['Correlation'])
top_corr = corr_df.sort_values('Abs_Correlation', ascending=False).head(10)
print(top_corr[['Feature_1', 'Feature_2', 'Correlation']])

# Variance analysis
print("\n3. FEATURE VARIANCE ANALYSIS:")
print("-" * 60)
variance_analysis = pd.DataFrame({
    'Feature': feature_cols,
    'Variance': [df_transformed[col].var() for col in feature_cols],
    'Std_Dev': [df_transformed[col].std() for col in feature_cols],
    'CV': [df_transformed[col].std() / abs(df_transformed[col].mean()) 
           if df_transformed[col].mean() != 0 else 0 
           for col in feature_cols]
})
variance_analysis = variance_analysis.sort_values('CV', ascending=False)
print("Features by Coefficient of Variation (Std/Mean):")
print(variance_analysis)

print("\n4. FEATURE CATEGORIES:")
print("-" * 60)

feature_categories = {
    'Soil': ['Clay', 'Sand', 'pH', 'BulkDensity', 'Nitrogen'],
    'Climate': ['Rain', 'Temp', 'LST', 'AtmosphericMoisture', 'SoilMoisture'],
    'Vegetation': ['NDVI', 'NDWI', 'green_fraction'],
    'Pollution': ['AOD', 'NO2', 'SO2'],
    'Topography': ['Elevation', 'Slope'],
    'Spatial': ['longitude', 'latitude']
}

for category, features in feature_categories.items():
    available = [f for f in features if f in feature_cols]
    print(f"\n{category} features ({len(available)}): {available}")

print("\n5. RECOMMENDED FEATURE SETS:")
print("-" * 60)

print("\n📊 MINIMAL SET (Core features only):")
minimal_features = [
    'Rain', 'Temp', 'NDVI', 'Clay', 'Nitrogen', 'pH', 
    'Elevation', 'Slope', 'SoilMoisture'
]
available_minimal = [f for f in minimal_features if f in feature_cols]
print(f"   {available_minimal}")
print(f"   Count: {len(available_minimal)}")

print("\n📊 STANDARD SET (Balanced coverage):")
standard_features = [
    # Climate
    'Rain', 'Temp', 'LST', 'AtmosphericMoisture', 'SoilMoisture',
    # Vegetation
    'NDVI', 'NDWI', 'green_fraction',
    # Soil
    'Clay', 'Nitrogen', 'pH', 'BulkDensity',
    # Topography
    'Elevation', 'Slope',
    # Pollution
    'AOD', 'NO2', 'SO2'
]
available_standard = [f for f in standard_features if f in feature_cols]
print(f"   {available_standard}")
print(f"   Count: {len(available_standard)}")

print("\n📊 FULL SET (All features):")
full_features = [f for f in feature_cols if f not in ['longitude', 'latitude']]
print(f"   {full_features}")
print(f"   Count: {len(full_features)}")

print("\n📊 SPATIAL SET (Including location):")
spatial_features = full_features + ['longitude', 'latitude']
print(f"   {spatial_features}")
print(f"   Count: {len(spatial_features)}")

# Domain-specific recommendations
print("\n6. DOMAIN-SPECIFIC RECOMMENDATIONS:")
print("-" * 60)

print("\n🌾 For CROP YIELD/AGRICULTURE prediction:")
print("   Priority: Rain, Temp, SoilMoisture, NDVI, Clay, Nitrogen, pH")

print("\n🌱 For VEGETATION HEALTH/NDVI prediction:")
print("   Priority: Rain, Temp, SoilMoisture, NDWI, Clay, Nitrogen, Elevation")

print("\n💧 For WATER STRESS/DROUGHT prediction:")
print("   Priority: Rain, SoilMoisture, NDWI, Temp, LST, AtmosphericMoisture")

print("\n🏭 For AIR QUALITY prediction:")
print("   Priority: AOD, NO2, SO2, Temp, AtmosphericMoisture, green_fraction")

print("\n🗺️ For LAND USE classification:")
print("   Priority: NDVI, green_fraction, Elevation, Slope, longitude, latitude")


FEATURE IMPORTANCE ANALYSIS

Available features for modeling: 20
['AOD', 'AtmosphericMoisture', 'BulkDensity', 'Clay', 'Elevation', 'LST', 'NDVI', 'NDWI', 'NO2', 'Nitrogen', 'Rain', 'SO2', 'Sand', 'Slope', 'SoilMoisture', 'Temp', 'green_fraction', 'pH', 'longitude', 'latitude']

1. DATA QUALITY CHECK:
------------------------------------------------------------
⚠ Features with NaN values:
NO2    2
dtype: int64

2. INTER-FEATURE CORRELATION MATRIX:
------------------------------------------------------------
Top 10 strongest correlations (excluding self-correlation):

               Feature_1       Feature_2  Correlation
99                  NDVI            NDWI    -0.915431
62                  Clay            Sand    -0.897186
27   AtmosphericMoisture            Rain     0.858151
93                   LST    SoilMoisture    -0.781757
168                 Sand        latitude     0.762691
94                   LST            Temp     0.745205
175         SoilMoisture            Temp    -0.

In [12]:
print("\n" + "="*80)
print("TIME-BASED TRAIN-TEST SPLIT")
print("="*80)

# Ensure date is parsed
df_transformed['date'] = pd.to_datetime(df_transformed['date'])

# Check temporal distribution
print("1. TEMPORAL DISTRIBUTION:")
print("-" * 60)
temporal_summary = df_transformed.groupby(['year', 'month']).agg({
    'date': 'count'
}).rename(columns={'date': 'records'})
print(temporal_summary)

total_records = len(df_transformed)
print(f"\nTotal records: {total_records}")

# Distribution by year
year_dist = df_transformed.groupby('year').size()
print(f"\nRecords by year:")
for year, count in year_dist.items():
    print(f"  {year}: {count:,} ({count/total_records*100:.1f}%)")

print("\n2. TIME-BASED SPLIT OPTIONS:")
print("-" * 60)

# Option 1: Simple year-based split
print("\n📅 OPTION 1: Simple Year-based Split")
print("   Train: 2021 + 2022")
print("   Test:  2023")

train_mask_1 = df_transformed['year'].isin([2021, 2022])
test_mask_1 = df_transformed['year'] == 2023

train_size_1 = train_mask_1.sum()
test_size_1 = test_mask_1.sum()

print(f"   Train size: {train_size_1:,} ({train_size_1/total_records*100:.1f}%)")
print(f"   Test size:  {test_size_1:,} ({test_size_1/total_records*100:.1f}%)")

# Option 2: Train-Val-Test split
print("\n📅 OPTION 2: Train-Validation-Test Split")
print("   Train:      2021")
print("   Validation: 2022")
print("   Test:       2023")

train_mask_2 = df_transformed['year'] == 2021
val_mask_2 = df_transformed['year'] == 2022
test_mask_2 = df_transformed['year'] == 2023

train_size_2 = train_mask_2.sum()
val_size_2 = val_mask_2.sum()
test_size_2 = test_mask_2.sum()

print(f"   Train size:      {train_size_2:,} ({train_size_2/total_records*100:.1f}%)")
print(f"   Validation size: {val_size_2:,} ({val_size_2/total_records*100:.1f}%)")
print(f"   Test size:       {test_size_2:,} ({test_size_2/total_records*100:.1f}%)")

# Option 3: Specific date cutoffs
print("\n📅 OPTION 3: Specific Date Cutoffs")
print("   Train:      Before 2022-07-01")
print("   Validation: 2022-07-01 to 2022-12-31")
print("   Test:       After 2022-12-31")

train_mask_3 = df_transformed['date'] < '2022-07-01'
val_mask_3 = (df_transformed['date'] >= '2022-07-01') & (df_transformed['date'] < '2023-01-01')
test_mask_3 = df_transformed['date'] >= '2023-01-01'

train_size_3 = train_mask_3.sum()
val_size_3 = val_mask_3.sum()
test_size_3 = test_mask_3.sum()

print(f"   Train size:      {train_size_3:,} ({train_size_3/total_records*100:.1f}%)")
print(f"   Validation size: {val_size_3:,} ({val_size_3/total_records*100:.1f}%)")
print(f"   Test size:       {test_size_3:,} ({test_size_3/total_records*100:.1f}%)")

# Option 4: Percentage-based with time order
print("\n📅 OPTION 4: Percentage-based (Chronological)")
print("   Train:      First 60%")
print("   Validation: Next 20%")
print("   Test:       Last 20%")

# Sort by date first
df_sorted = df_transformed.sort_values('date').reset_index(drop=True)
n_train = int(0.6 * len(df_sorted))
n_val = int(0.2 * len(df_sorted))

train_mask_4 = df_sorted.index < n_train
val_mask_4 = (df_sorted.index >= n_train) & (df_sorted.index < n_train + n_val)
test_mask_4 = df_sorted.index >= n_train + n_val

print(f"   Train size:      {train_mask_4.sum():,} (60%)")
print(f"   Validation size: {val_mask_4.sum():,} (20%)")
print(f"   Test size:       {test_mask_4.sum():,} (20%)")
print(f"   Train dates: {df_sorted[train_mask_4]['date'].min()} to {df_sorted[train_mask_4]['date'].max()}")
print(f"   Val dates:   {df_sorted[val_mask_4]['date'].min()} to {df_sorted[val_mask_4]['date'].max()}")
print(f"   Test dates:  {df_sorted[test_mask_4]['date'].min()} to {df_sorted[test_mask_4]['date'].max()}")

print("\n3. IMPLEMENTATION CODE:")
print("-" * 60)

print("""
# Choose one of the options above, here's the code for OPTION 1 (recommended):

# Define features (use your chosen feature set)
feature_columns = [
    'Rain', 'Temp', 'LST', 'AtmosphericMoisture', 'SoilMoisture',
    'NDVI', 'NDWI', 'green_fraction',
    'Clay', 'Nitrogen', 'pH', 'BulkDensity',
    'Elevation', 'Slope',
    'AOD', 'NO2', 'SO2'
]

# Define target (replace with your actual target)
# target_column = 'your_target'

# Create masks
train_mask = df_transformed['year'].isin([2021, 2022])
test_mask = df_transformed['year'] == 2023

# Split data
X_train = df_transformed.loc[train_mask, feature_columns]
X_test = df_transformed.loc[test_mask, feature_columns]

# y_train = df_transformed.loc[train_mask, target_column]
# y_test = df_transformed.loc[test_mask, target_column]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
# print(f"y_train shape: {y_train.shape}")
# print(f"y_test shape: {y_test.shape}")

# Optional: Save train/test indices for reproducibility
train_indices = df_transformed[train_mask].index.tolist()
test_indices = df_transformed[test_mask].index.tolist()
""")

print("\n4. SCALING CODE (if needed):")
print("-" * 60)

print("""
# For models that require scaling (Linear, SVM, Neural Networks):

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Option A: StandardScaler (most common)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame to keep feature names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_columns, index=X_test.index)

# Option B: MinMaxScaler (for Neural Networks)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Option C: RobustScaler (for data with outliers - your case)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# For tree-based models (Random Forest, XGBoost, LightGBM):
# No scaling needed! Use X_train and X_test directly
""")

print("\n5. CROSS-VALIDATION FOR TIME-SERIES:")
print("-" * 60)

print("""
# For hyperparameter tuning, use TimeSeriesSplit instead of regular CV

from sklearn.model_selection import TimeSeriesSplit

# Create time series CV splitter
tscv = TimeSeriesSplit(n_splits=5)

# Use with GridSearchCV or cross_val_score
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)

# This ensures no data leakage (always trains on past, validates on future)
scores = cross_val_score(model, X_train, y_train, cv=tscv, 
                        scoring='neg_mean_squared_error')

print(f"CV RMSE scores: {np.sqrt(-scores)}")
print(f"Mean RMSE: {np.sqrt(-scores.mean()):.4f} (+/- {np.sqrt(scores.std()):.4f})")
""")

print("\n" + "="*80)
print("RECOMMENDED APPROACH:")
print("="*80)
print("""
✅ Use OPTION 1 for simplicity:
   - Train on 2021 + 2022
   - Test on 2023
   - 70/30 split is reasonable

✅ If you need validation set for hyperparameter tuning:
   - Use TimeSeriesSplit CV on the training set
   - Don't create separate validation set from test year

✅ For final evaluation:
   - Train on all 2021+2022 data with best hyperparameters
   - Evaluate once on 2023 test set
   - Report metrics on test set only

⚠️  Do NOT use random split - it will leak future information!
""")



TIME-BASED TRAIN-TEST SPLIT
1. TEMPORAL DISTRIBUTION:
------------------------------------------------------------
            records
year month         
2021 0.0        805
     1.0        805
     2.0        805
     3.0        805
     4.0        785
     5.0        710
     6.0          3
     7.0         63
     8.0        503
     9.0        263
     10.0       215
     11.0       805
2022 12.0       805
     13.0       805
     14.0       805
     15.0       805
     16.0       805
     17.0       805
     18.0        28
     19.0       199
     20.0       755
     21.0       805
     22.0       805
     23.0       805
2023 24.0       805
     25.0       805
     26.0       805
     27.0       805
     28.0       805
     29.0       783
     31.0       652
     32.0       373
     33.0       805
     34.0       805
     35.0       805

Total records: 23042

Records by year:
  2021: 6,567 (28.5%)
  2022: 8,227 (35.7%)
  2023: 8,248 (35.8%)

2. TIME-BASED SPLIT OPTIONS:
--------

In [13]:
df_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 23042 entries, 0 to 23041
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   system:index         23042 non-null  str           
 1   AOD                  23042 non-null  float64       
 2   AtmosphericMoisture  23042 non-null  float64       
 3   BulkDensity          23042 non-null  float64       
 4   Clay                 23042 non-null  float64       
 5   Elevation            23042 non-null  int64         
 6   LST                  23042 non-null  float64       
 7   NDVI                 23042 non-null  float64       
 8   NDWI                 23042 non-null  float64       
 9   NO2                  23040 non-null  float64       
 10  Nitrogen             23042 non-null  float64       
 11  Rain                 23042 non-null  float64       
 12  SO2                  23042 non-null  float64       
 13  Sand                 23042 non-null  float